# Objectif 

L'objectif de ce notebook est de modéliser la propagation dans la zone de la campagne de mesure à Groix dans le cadre du projet Fiberscope. En particulier, on souhaite : 

* Modéliser la réponse impulsionnelle du canal dans la gamme de fréquence d'intérêt, en déduire le temps de réverbération et comparer aux valeurs proposées par Myriam L. dans le rapport préliminaire (modélisation Bellhop)

* Tester différents paramètres du signal source pour identifier la configuration la plus efficace pour mener des essais de localisation à partir du vecteur de RTF. 

Dans le rapport, 6 profils sont considérés. On considère dans un premier temps le profil 1 reliant le point T5 à l'OBS 3. Le guide d'onde est modélisé par un guide de Pekeris dont les propriétés du sédiment sont données par le profil équivalent en distance. 

Données utilisées : 

* Bathymétrie : GEBCO 2021
* SSP 

Coordonnées des points (d'après la fiche prévisionnelle des expérimentations envisagées) -> pos_dm.csv 


In [ ]:
import os
import sys
import arlpy
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd 

sys.path.append(r"C:\Users\baptiste.menetrier\Desktop\devPy\phd")

# Load usefull functions
import source.global_constants as g
from publication.publication_figure import PubFigure, LargeFigure, SmallFigure
from propa.ideal_waveguide import (
    psi,
    psi_normalised,
    h,
    field,
    plot_tl,
    nb_propagating_modes,
    print_arrivals,
)

from propa.kraken_toolbox.src.kraken_testcase import (
    KrakenTestCase,
    DomainProperties,
    SourceProperties,
    ReceiverProperties,
    KrakenProperties,
)
from propa.kraken_toolbox.src.kraken_env import (
    KrakenTopHalfspace,
    KrakenMedium,
    KrakenBottomHalfspace,
    KrakenAttenuation,
    KrakenField,
    Bathymetry,
)
from propa.kraken_toolbox.src.kraken_manager import KrakenManager
from propa.kraken_toolbox.plot_utils import plotshd, plotmode, plotmode_several_freqs
from propa.kraken_toolbox.utils import default_nb_rcv_z
from signals.AcousticComponent import AcousticSource
from source.signal_generator import SignalGenerator
from source.ssp_profiles import SSPProfile
from misc import mult_along_axis

from get_data.cmems import load_data_from_cmems as cmems
from get_data.bathymetry import bathy_profile_extraction as bpe
pfig = PubFigure()

In [ ]:
folder_root = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\propa\rtf\rtf_estimation\xp_fiberscope_groix"
tc_root_dir = (
    r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\propa\kraken_toolbox\testcases"
)
img_folder_path = os.path.join(folder_root, "img")
data_folder_path = os.path.join(folder_root, "data")

if not os.path.exists(img_folder_path):
    os.makedirs(img_folder_path)

if not os.path.exists(data_folder_path):
    os.makedirs(data_folder_path)

input_data_root = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\data"
bathy_fpath = os.path.join(input_data_root, "bathy", "GEBCO_2021_sub_ice_topo.nc")
# # Dataset path
# tf_demo_fpath = os.path.join(data_folder_path, "tf_demo.nc")
# tf_perf_fpath = os.path.join(data_folder_path, "tf_perf.nc")

# sig_fpath = os.path.join(data_folder_path, "received_signals.nc")

# Etape 1 : chargement et mise en forme des données nécessaires 

In [ ]:
# Load coords of the experiment
df_coords = pd.read_csv(os.path.join(data_folder_path, "pos_deg.csv"), index_col=0)
print(f"OBS 3: lon = {df_coords.loc['obs3'].lon}, lat = {df_coords.loc['obs3'].lat}")
dlat_box = 0.1
dlon_box = 0.1

### Bathy

In [ ]:
# Load bathy data 
ds_bathy = xr.open_dataset(bathy_fpath)

In [ ]:
ds_bathy

In [ ]:
# Slice data to get the area of interest
ds_bathy = ds_bathy.sel(lat=slice(df_coords.loc["obs3"].lat - dlat_box, df_coords.loc["obs3"].lat + dlat_box), lon=slice(df_coords.loc["obs3"].lon - dlon_box, df_coords.loc["obs3"].lon + dlon_box))

In [ ]:
# Plot elevation
plt.figure()
ds_bathy.elevation.plot()
plt.scatter(df_coords.loc["obs3"].lon, df_coords.loc["obs3"].lat, color="red", label="OBS_3")
plt.scatter(df_coords.loc["t5"].lon, df_coords.loc["t5"].lat, color="blue", label="T5")

# Add contours
plt.contour(ds_bathy.lon, ds_bathy.lat, ds_bathy.elevation, levels=[-0], colors='black')

In [ ]:
# Extract bathy along the path between OBS3 and T5
dr = 200
# Rename variable to suit function input elevation -> bathymetry
ds_bathy = ds_bathy.rename({"elevation": "bathymetry"})

range_along_profile, bathymetry_profile = bpe.extract_bathy_profile(
    xr_bathy=ds_bathy,
    start_lat=df_coords.loc["t5"].lat,
    start_lon=df_coords.loc["t5"].lon,
    stop_lat=df_coords.loc["obs3"].lat,
    stop_lon=df_coords.loc["obs3"].lon,
    range_resolution=dr,
)
# Set positive bathymetry down
bathymetry_profile = -bathymetry_profile

In [ ]:
# Plot extracted profile
plt.figure()
plt.plot(range_along_profile / 1e3, bathymetry_profile)
plt.xlabel("Range [km]")
plt.ylabel("Depth [m]")
plt.title("Extracted bathymetry profile between T5 and OBS3")
plt.ylim(0, np.max(bathymetry_profile) * 1.1)
plt.gca().invert_yaxis()

In [ ]:
# Dummy testcase to init directories
name = "xr_fiberscope_groix_p1"
dummmy_tc = KrakenTestCase(
    name=name,
    root_dir=tc_root_dir,
)

In [ ]:
# Define bathy object for kraken
bathy_arr = np.array([range_along_profile * 1e-3, bathymetry_profile]).T
# Convert to datafram
df_bathy = pd.DataFrame(bathy_arr)
# Save to csv
bathy_fpath = os.path.join(data_folder_path, "bathy.csv")
df_bathy.to_csv(bathy_fpath, index=False, header=False)


bathy = Bathymetry(
    data_file=bathy_fpath,
    interpolation_method="linear",
    units="km",
)

### SSP 

In [ ]:
download_cmems_data = False

# dataset_id = "cmems_mod_glo_phy_anfc_0.083deg_PT1H-m"
# dataset_version = "202406"
# start_datetime = "2022-09-15T00:00:00"
# end_datetime = "2022-09-16T00:00:00"

dataset_id = "cmems_mod_glo_phy_my_0.083deg_P1D-m"
dataset_version = "202311"
start_datetime = "2020-09-15T00:00:00"
end_datetime = "2020-09-16T00:00:00"

# Set file name
fname = f"cmems_thetao_so_{dataset_id.split('_')[-1]}_{start_datetime[:10]}_{end_datetime[:10]}.nc"


if download_cmems_data: 

    data_request = dict(
        dataset_id=dataset_id,
        dataset_version="202406",
        variables=[
            "so",
            "thetao",
        ],
        minimum_longitude=df_coords.loc["obs3"].lon - dlon_box,
        maximum_longitude=df_coords.loc["obs3"].lon + dlon_box,
        minimum_latitude=df_coords.loc["obs3"].lat - dlat_box,
        maximum_latitude=df_coords.loc["obs3"].lat + dlat_box,
        start_datetime=start_datetime,
        end_datetime=end_datetime,
        minimum_depth=0,
        maximum_depth=100,
        output_dir=data_folder_path,
        output_filename=fname,
        force_download=True,
    )

    ds_cmems = cmems.load_data(data_request)

else:
    pass

# Load existing file
fpath = os.path.join(data_folder_path, fname)
ds_cmems = xr.open_dataset(fpath) 

In [ ]:
ds_cmems

In [ ]:
# Plot spatial coverage
plt.figure()
ds_cmems.thetao.isel(depth=0, time=0).plot()

# Add points
plt.scatter(df_coords.loc["obs3"].lon, df_coords.loc["obs3"].lat, color="red", label="OBS_3")
plt.scatter(df_coords.loc["t5"].lon, df_coords.loc["t5"].lat, color="blue", label="T5")
plt.legend()

# Add bathy contours
plt.contour(ds_bathy.lon, ds_bathy.lat, ds_bathy.bathymetry, levels=[0], colors='black')

In [ ]:
# Extract data profile at the location of the emitter (T5)
ds_cmems_t5 = ds_cmems.sel(longitude=df_coords.loc["t5"].lon, latitude=df_coords.loc["t5"].lat, method="nearest")
# Extrat data profile at the location of the receiver (OBS3)
ds_cmems_obs3 = ds_cmems.sel(longitude=df_coords.loc["obs3"].lon, latitude=df_coords.loc["obs3"].lat, method="nearest")

In [ ]:
# Plot profiles
fig, axs = plt.subplots(1, 2, figsize=(10, 6), sharey=True)
ds_cmems_t5.thetao.isel(time=0).plot(y="depth", label="T-T5", ax=axs[0])
ds_cmems_obs3.thetao.isel(time=0).plot(y="depth", label="T-OBS3", ax=axs[0])

axs[0].set_title("Temperature profile")
ds_cmems_t5.so.isel(time=0).plot(y="depth", label="S-T5", ax=axs[1])
ds_cmems_obs3.so.isel(time=0).plot(y="depth", label="S-OBS3", ax=axs[1])

axs[1].set_title("Salinity profile")
axs[0].invert_yaxis()
axs[0].set_ylabel("")
axs[1].set_ylabel("")
fig.supylabel("Depth [m]")
axs[0].legend()
axs[1].legend()

# ds_cmems.thetao_std.isel(time=0, lon=0).plot(y="depth", label="Std")

In [ ]:
# Get associated ssp 
c_t5 = arlpy.uwa.soundspeed(temperature=ds_cmems_t5.thetao.isel(time=0).values, salinity=ds_cmems_t5.so.isel(time=0).values, depth=ds_cmems_t5.depth.values)
c_obs3 = arlpy.uwa.soundspeed(temperature=ds_cmems_obs3.thetao.isel(time=0).values, salinity=ds_cmems_obs3.so.isel(time=0).values, depth=ds_cmems_obs3.depth.values)

In [ ]:
# Put c in a dedicated xarray dataset
ds_ssp_t5 = xr.Dataset(
    {
        "c": (("depth"), c_t5)
    },
    coords={
        "depth": ds_cmems_t5.depth.values
    }
)
ds_ssp_obs3 = xr.Dataset(
    {
        "c": (("depth"), c_obs3,)
    },
    coords={
        "depth": ds_cmems_obs3.depth.values
    }
)

In [ ]:
# Plot celerity profiles
plt.figure(figsize=(6, 8))
ds_ssp_t5.c.plot(y="depth", label="T-T5")
ds_ssp_obs3.c.plot(y="depth", label="T-OBS3")
plt.gca().invert_yaxis()
plt.xlabel("Sound speed [m/s]")
plt.ylabel("Depth [m]")


In [ ]:
use_xp_ssp = True
fname = "svp_1159_04112017.csv"
if use_xp_ssp: 
    fpath = os.path.join(data_folder_path, fname)
    df_ssp_xp = pd.read_csv(fpath)

In [ ]:
df_ssp_xp

In [ ]:
# Define xarray dataset
ds_ssp_xp = xr.Dataset(
    {
        "c": (("depth"), df_ssp_xp.ssp.values)
    },
    coords={
        "depth": df_ssp_xp.d.values
    }
)
ds_ssp_xp.attrs["date"] = fname.split("_")[1]

In [ ]:
# Plot celerity profiles
plt.figure(figsize=(6, 8))
ds_ssp_xp.c.plot(y="depth", label=ds_ssp_xp.attrs["date"])
# reverse y axis
plt.gca().invert_yaxis()

# Etape 2 : définition du cas test 

## Paramètres du guide d'onde de Pekeris

D'après le rapport du SHOM le tracé de rayon est effectué avec un sédiment sable (cf P;14 en dessous de la figure 17). 

In [ ]:
# Waveguide geometry
depth = np.ceil(bathy.bathy_depth.max())  # m
zmin = 1
zmax = depth
max_range_km = bathy.bathy_range.max()
rmin = 0
rmax = max_range_km * 1e3

print(f"Waveguide depth: {depth} m")
print(f"Waveguide range: {max_range_km} km")

# Sediment
bott_props = g.sand_properties
print(f"Sediment properties: {bott_props}")

## Paramètres de la source 

In [ ]:
src_depth = 5 # m
src_min_freq = 200 # Hz
src_max_freq = 1000 # Hz
src_fs = 2000  # Hz
src_signal_duration = 10  # s

# Create dummy signal to make it easy to run kraken simulation
fc = src_max_freq / 2  # Hz
sg = SignalGenerator()
s, t = sg.pulse(T=src_signal_duration, fc=fc, fs=src_fs, t0=0)

s = sg.normalize_sig(s, normalize="max")
# Plot time serie
sg.plot_signal(t, s)
plt.gca().set_xlim([0, 0.1])

src = AcousticSource(
    signal=s,
    time=t,
    name="Pulse",
    waveguide_depth=depth,
    window=None,
    nfft=2 ** int(np.log2(s.size) + 1),
)

## Paramètres généraux et objects KrakenEnv

In [ ]:
# Common properties
title = "Fiberscope Groix - Profile P1"

In [ ]:
# Set domain properties
domain_properties = DomainProperties(
    zmin=zmin, zmax=zmax, rmin=rmin, rmax=rmax, unit="m"
)

In [ ]:
# Set source properties
src_properties = SourceProperties(
    src_type="point_source", src_depth=src_depth, freq=src.kraken_freq
)

In [ ]:
# Set receiver properties : needs to cover the whole water domain
rcv_z_min = zmin
rcv_z_max = zmax

# Number of receiver depths / ranges (flp file) : sufficient resolution for later use (can be easily downsampled afterwards)
dr = 50
dz = 5
nr_flp = int(rmax / dr) + 1
nz_flp = int(rcv_z_max / dz) + 1

rcv_properties = ReceiverProperties(
    zmin=rcv_z_min, zmax=rcv_z_max, rmin=rmin, rmax=rmax, unit="m"
)

In [ ]:
# Set layer properties
nmedia = 2

top_hs = KrakenTopHalfspace(
    boundary_condition="vacuum",
    halfspace_properties=None,
    twersky_scatter_properties=None,
)

bott_hs = KrakenBottomHalfspace(
    boundary_condition="acousto_elastic",
    sigma=0.0,
    halfspace_properties=bott_props,
    fmin=src.kraken_freq.min(),
    alpha_wavelength=10,
)

In [ ]:
# Set attenuation properties
att = KrakenAttenuation(units="dB_per_wavelength", use_volume_attenuation=False)

In [ ]:
# Set SSP object for kraken
if use_xp_ssp:
    z_ssp = df_ssp_xp.d.values
    c = df_ssp_xp.ssp.values
else:
    z_ssp = ds_cmems_obs3.depth.values
    c = ds_ssp_obs3.c.values

ssp = SSPProfile(z=z_ssp, c=c)

In [ ]:
# Create the medium = water column layer
medium = KrakenMedium(
    ssp_interpolation_method="C_linear",
    z_ssp=ssp.z,
    c_p=ssp.c,
    c_s=0.0,
    rho=g.rho_w,
    a_p=0.0,
    a_s=0.0,
    nmesh=0,
    sigma=0.0,
)

In [ ]:
# Set field properties
bott_hs.derive_sedim_layer_max_depth(domain_properties.zmax_m)
max_rcv_depth = bott_hs.sedim_layer_max_depth
n_rcv_z = default_nb_rcv_z(
    fmax=src.kraken_freq.max(), max_depth=max_rcv_depth, n_per_l=10
)

min_phase_speed = 1000
max_phase_speed = 20000

field = KrakenField(
    phase_speed_limits=[min_phase_speed, max_phase_speed],
    src_depth=[src_properties.depth],
    n_rcv_z=n_rcv_z,
    rcv_z_min=0,
    rcv_z_max=max_rcv_depth,
    rcv_r_max=0.0,
)

In [ ]:
# Set kraken properties
kraken_properties = KrakenProperties(
    mode_coupling="coupled",
    mode_addition="coherent",
    n_mode=100,
    nr=nr_flp,
    nz=nz_flp,
    nmedia=nmedia,
    top_hs=top_hs,
    bott_hs=bott_hs,
    att=att,
    medium=medium,
    field=field,
)

In [ ]:
# Define testcase 
k_tc = KrakenTestCase(
    name=name,
    title=title,
    root_dir=tc_root_dir,
    bathy=bathy,
    domain_properties=domain_properties,
    src_properties=src_properties,
    rcv_properties=rcv_properties,
    kraken_properties=kraken_properties,
)

In [ ]:
# Kraken compute otpion
# run_kraken = False
run_kraken = True

## Calcul des fonctions de Green du guide d'onde étudié

In [38]:
if run_kraken:
    km = KrakenManager()
    pressure_field, field_pos = km.runkraken(
        env=k_tc.env, flp=k_tc.flp, frequencies=k_tc.src.freq
    )

## Sauvergarde des fonctions de transfert du guide d'onde 

In [ ]:
if run_kraken:
    # Store pressure field as netcdf using xarray
    pressure_field = np.squeeze(pressure_field)  # Remove singleton dimensions if any
    ds_tf = xr.Dataset(
        data_vars=dict(
            tf_real=(["f", "z", "r"], np.real(pressure_field)),
            tf_imag=(["f", "z", "r"], np.imag(pressure_field)),
        ),
        coords=dict(
            f=k_tc.src.freq,
            z=field_pos["r"]["z"],
            r=field_pos["r"]["r"],
        ),
        attrs=dict(
            title="Transfer functions for Pekeris waveguide",
            description="Transfer functions computed using Kraken for a Pekeris waveguide with short impulse response.",
            # date_created=np.datetime64("now"),
            note="Dataset for demo purpose: high spatial resolution and full range coverage.",
            type="demo",
            fs=src_fs,
            signal_duration=src_signal_duration,

        ),
    )
    # Save to netcdf
    ds_tf.to_netcdf(tf_demo_fpath)
else:
    # Load transfer functions from netcdf
    ds_tf = xr.open_dataset(tf_demo_fpath)

## Visualisation des modes du guide d'onde 

In [ ]:
lfig = LargeFigure()

f_visu = 25  # Frequency to visualize
mode_fpath = os.path.join(k_tc.io_files_dir, k_tc.env.filename)
plotmode(
    mode_fpath,
    freq=[f_visu],
    modes=[1, 2, 12, 13],
    bathy_depth=k_tc.bathy.bathy_depth[0],
    normalize_mode=True,
)

fig = plt.gcf()
plt.ylim([depth+500, 0])
fig.suptitle(f"f = {f_visu} Hz")

# Save figure as pdf
fpath = os.path.join(img_folder_path, f"{name}_4modes.pdf")
plt.savefig(fpath, dpi=300)

In [ ]:
mode_fpath = os.path.join(k_tc.io_files_dir, k_tc.env.filename)

lfig = LargeFigure(legend_fontsize=16)

f_visu = np.array([5, 10, 45])  # Frequencies to visualize
modes=[1, 2, 12, 13]
plotmode_several_freqs(
    mode_fpath,
    freq=f_visu,
    modes=modes,
    bathy_depth=k_tc.bathy.bathy_depth[0],
    normalize_mode=True,
)

fig = plt.gcf()
plt.ylim([depth + 500, 0])
# fig.suptitle(f"f = {f_visu} Hz")
fig.suptitle("")


# Save figure as pdf
f_name = "Hz_".join([str(f) for f in f_visu]) + "Hz"
mode_name = "_".join([str(m) for m in modes])
fpath = os.path.join(img_folder_path, f"{name}_mode_{mode_name}_f_{f_name}.pdf")
plt.savefig(fpath, dpi=300)

In [ ]:
test_envdir = r"C:\Users\baptiste.menetrier\Desktop\devPy\phd\propa\kraken_toolbox\testcases\perekis_short_ir_waveguide_test\io_files"
test_envfilename = "perekis_short_ir_waveguide_test.env"
mode_fpath = os.path.join(test_envdir, test_envfilename)


lfig = LargeFigure(legend_fontsize=16)

f_visu = np.array([5, 10, 45])  # Frequencies to visualize
modes = [1, 2]
plotmode_several_freqs(
    mode_fpath,
    freq=f_visu,
    modes=modes,
    bathy_depth=k_tc.bathy.bathy_depth[0],
    normalize_mode=True,
)

fig = plt.gcf()
plt.ylim([depth + 500, 0])
# fig.suptitle(f"f = {f_visu} Hz")
fig.suptitle("")



### On peut comparer les angles associés aux modes à l'angle critique 

On a : 

$\theta_m = \arctan{\frac{k_{zm}}{k_{rm}}}$

et pour un guide d'onde de Pekeris (Cf Jensen et al 2011, p.355) :

$k_{rm} = \sqrt{k^2 - k_{zm}^2}$

et $k_{zm}$ est solution de l'équation transcendentale :

$\tan{k_{zm}D} = - \frac{i \rho_b k_{zm}} {\rho_w k_{zm,b}}$

In [ ]:
# Derive associated modes propagation angles 

# theta_m = np.atan(kzm / krm)

In [ ]:
# fréquence de coupure du mode m
m = 12
f_0m = ((m -0.5) * c_water) / (2 * depth * np.sqrt(1 - (c_water / c_sediment) ** 2))
print(f"Cut-off frequency of mode {m} : {f_0m:.2f} Hz")

## Visualisation des pertes à 3 fréquences 

Remarque : ici la source est placée à  1 m au dessus de la surface pour le calcul Kraken, l'objectif est d'exploiter la réciprocité pour considérée le signal reçu sur une antenne linéaire horizontal située 1 au dessus du fond.  

In [ ]:
lfig = LargeFigure(size=(16, 7.5))

# Plot tl at a single frequency
tf = ds_tf.tf_real + 1j * ds_tf.tf_imag  # (nf, nz, nr)
freq_plot = [5, 25, 45]
p_field = tf.sel(f=freq_plot, method="nearest").values
p_field[(p_field == 0) | np.isnan(p_field)] = 1e-20
tl = -20 * np.log10(np.abs(p_field))
tlmax = np.percentile(tl, 95)
tlmin = np.percentile(tl, 1) - 15

# Plot TL for each number of modes
fig, axs = plt.subplots(len(freq_plot), figsize=(10, 10), sharex=True)
abcd_labels = ["a", "b", "c"]
for i, fp in enumerate(freq_plot):
    p_field = p_field[i, ...]
    title = f"f = {fp} Hz"

    # Plot TL
    im = axs[i].pcolormesh(
        tf.r * 1e-3, tf.z, tl[i], cmap="jet_r", vmin=tlmin, vmax=tlmax, rasterized=True
    )

    # Add source position
    if src_depth is not None:
        axs[i].scatter(
            0,
            src_depth,
            color="k",
            marker="o",
            s=150,
        )
    axs[i].set_title(title)
    axs[i].invert_yaxis()

    # Add a, b, c labels
    axs[i].text(
        0.95,
        1.05,
        f"({abcd_labels[i]})",
        transform=axs[i].transAxes,
        fontsize=25,
        fontweight="bold",
        va="bottom",
    )

fig.supxlabel("Range [km]")
fig.supylabel("Depth [m]")

# Add common colorbar
cbar = fig.colorbar(im, ax=axs, orientation="vertical", pad=0.05, aspect=40)
cbar.set_label("TL [dB]")

# Save figure as pdf
fpath = os.path.join(img_folder_path, f"{name}_tl_3freqs.pdf")
plt.savefig(fpath)

## Visualisation du signal propagé

In [ ]:
run_this_section = False
# run_this_section = True

In [ ]:
rmin_visu = 30 * 1e3  # Minimum range for visualization
rmax_visu = 30 * 1e3  # Maximum range for visualization
ds_tf_pulse = ds_tf.sel(r=slice(rmin_visu, rmax_visu))  # Select receivers at 30 km range
rcv_range = ds_tf_pulse.r.values

delays = rcv_range / g.c0
rcv_depth = np.linspace(k_tc.flp.rcv_z_min, k_tc.flp.rcv_z_max, k_tc.flp.n_rcv_z)

In [ ]:
if run_this_section:
    # Source spectrum
    Sf = src.positive_spectrum
    fmin = np.min(ds_tf_pulse.f.values)
    idx_first_freq = np.argmin(np.abs(src.positive_freq - fmin))
    Sf = Sf[idx_first_freq:]

    # Derive delay for each receiver
    tau_rcv = ds_tf_pulse.r.min().values / g.c0
    # tau_rcv = target_range / g.c0

    tf = ds_tf_pulse.tf_real + 1j * ds_tf_pulse.tf_imag  # (nf, nz, nr)

    # Derive received spectrum (Y = SH)
    k0 = 2 * np.pi * ds_tf_pulse.f.values / g.c0
    norm_factor = np.exp(1j * k0) / (4 * np.pi)

    # # Derive delay factor to take into account the propagation time
    delay_rcv = np.exp(1j * 2 * np.pi * tau_rcv * ds_tf_pulse.f.values)  # (nf,)

    y_f = mult_along_axis(tf, Sf * norm_factor * delay_rcv, axis=0)

    nfft_inv = (
        4 * src.nfft
    )  # according to Jensen et al. (2000) p.616 : dt < 1 / (8 * fmax) for visual inspection of the propagated pulse
    T_tot = 1 / src.df
    dt = T_tot / nfft_inv
    time_vector = np.arange(0, T_tot, dt)

    # FFT inv to get signal
    y_t = np.fft.irfft(y_f, axis=0, n=nfft_inv)  # (nt, nz, nr)
    y_t = np.real(y_t)  # Keep only real part

    # Build dataset to save
    ds_sig = xr.Dataset(
        coords=dict(
            t=time_vector,
            z=ds_tf_pulse.z,
            r=ds_tf_pulse.r,
        ),
        data_vars=dict(
            s=(["t", "z", "r"], y_t),
        ),
    )

    # Save dataset
    ds_sig.to_netcdf(sig_fpath)
else:
    # Load dataset
    ds_sig = xr.open_dataset(sig_fpath)

In [ ]:
range_plot = np.array([30000])
dz = 100
depth_plot = np.arange(dz, 1000, dz)  # Depths to plot

# Scale for visualization
max_amplitude = ds_sig.s.max().values
alpha_dilatation = 1 * dz 
ds_sig["s"] = ds_sig.s / max_amplitude * alpha_dilatation

# Roll over time axis to start at zero
tau_roll = (
    range_plot[0] - ds_sig.r.min().values
) / g.c0  # Roll time axis to start at zero
ts = ds_sig.t.diff("t").values[0]
idx_tau_roll = int(tau_roll / ts)
sig_roll = ds_sig.s.roll(t=-idx_tau_roll, roll_coords=False)

z_offset = depth_plot[1] - depth_plot[0]
for ir, r in enumerate(range_plot):
    plt.figure(figsize=(8, 12))
    for iz, z in enumerate(depth_plot):
        sig = sig_roll.sel(z=z, r=r, method="nearest") + (iz + 1) * z_offset
        if z == src_depth:
            sig.plot(color="r", label="Source depth")
        else:
            sig.plot(color="k")

    # Revert y-axis
    ax = plt.gca()
    ax.invert_yaxis()
    # plt.xlim([0, 1])
    plt.xlabel(f"Time  t - r/{g.c0} [s]")
    plt.ylabel("Depth [m]")
    plt.title(
        "",
    )
    # Save figure as pdf
    fpath = os.path.join(img_folder_path, f"{name}_received_signals_{range_plot[0]}m.pdf")
    plt.savefig(fpath)

In [ ]:
z_plot = 995
r_plot = 30 * 1e3

ts = ds_sig.t.diff("t").values[0]
tau_roll = (r_plot - ds_sig.r.min().values) / g.c0  # Roll time axis to start at zero
idx_tau_roll = tau_roll / ts
idx_tau_roll = idx_tau_roll.astype(int)

sig = ds_sig.s.sel(r=r_plot, z=z_plot, method="nearest")
sig = sig / sig.max().values
sig_roll = sig.roll(t=-idx_tau_roll, roll_coords=False)
sig_roll.plot(color="k")
plt.xlabel(f"Time  t - r/{g.c0} [s]")
plt.ylabel("Amplitude")

In [ ]:
# Derive prms
p = sig_roll
t_win = 100
# print(f"Window size for RMS: {t_win} samples")
print(f"Window duration for RMS: {t_win * ts:.2f} s")
p2_roll = (p**2).rolling(t=t_win, center=True).mean()  
p_rms = np.sqrt(p2_roll)

# Plot p_rms
plt.figure()
p_rms.plot(color="k")
plt.xlabel(f"Time  t - r/{g.c0} [s]")
plt.ylabel("RMS pressure")
plt.title("")
# plt.xlim([0, 4])  # Cropp the end to avoid wrapped around signal artefacts

In [ ]:
# Derive SPL
# p_rms /= p_rms.max().values
spl = 20 * np.log10(p_rms / g.p0)

# Plot spl
spl.plot(color="k")

# Add threshold line
threshold = -30  # Threshold in dB
th = np.max(spl).values + threshold
plt.axhline(
    y=th,
    color="r",
    linestyle="--",
    linewidth=1,
    label=f"{threshold} dB",
)

# plt.xlim([0, 4])  # Cropp the end to avoid wrapped around signal artefacts
plt.xlabel(f"Time  t - r/{g.c0} [s]")
plt.ylabel("SPL [dB re 1uPa]")
plt.legend()
plt.title("")

# # Save figure as pdf
# fpath = os.path.join(img_folder_path, f"{name}_ir_spl_{r_plot}m.pdf")
# plt.savefig(fpath)

# Visualisation de la réponse impulsionnelle

In [ ]:
run_this_section = False
# run_this_section = True

In [ ]:
rmin_visu = 20 * 1e3  # Minimum range for visualization
rmax_visu = 40 * 1e3  # Maximum range for visualization
dr_visu = 5000  # Range step for impulse response visualization
r_visu_ir = np.arange(
    rmin_visu, rmax_visu, dr_visu
)  # Range for impulse response visualization
ds_tf_ir = ds_tf.sel(r=r_visu_ir, method="nearest")  # Select receivers at 30 km range
rcv_range = ds_tf_ir.r.values

delays = rcv_range / g.c0
rcv_depth = np.linspace(k_tc.flp.rcv_z_min, k_tc.flp.rcv_z_max, k_tc.flp.n_rcv_z)

In [ ]:
ir_fpath = os.path.join(data_folder_path, "ir.nc")

if run_this_section:
    # Source spectrum
    Sf = src.positive_spectrum
    fmin = np.min(ds_tf_ir.f.values)
    idx_first_freq = np.argmin(np.abs(src.positive_freq - fmin))
    Sf = Sf[idx_first_freq:]

    # Set Sf to unity to get the impulse response 
    Sf[:] = 1

    # Derive delay for each receiver
    tau_rcv = ds_tf_ir.r.min().values / g.c0
    # tau_rcv = target_range / g.c0


    tf = ds_tf_ir.tf_real + 1j * ds_tf_ir.tf_imag  # (nf, nz, nr)

    # Derive received spectrum (Y = SH)
    k0 = 2 * np.pi * ds_tf_ir.f.values / g.c0
    norm_factor = np.exp(1j * k0) / (4 * np.pi)

    # # Derive delay factor to take into account the propagation time
    delay_rcv = np.exp(1j * 2 * np.pi * tau_rcv * ds_tf_ir.f.values)  # (nf,)

    y_f = mult_along_axis(tf, Sf * norm_factor * delay_rcv, axis=0)

    nfft_inv = (
        4 * src.nfft
    )  # according to Jensen et al. (2000) p.616 : dt < 1 / (8 * fmax) for visual inspection of the propagated pulse
    T_tot = 1 / src.df
    dt = T_tot / nfft_inv
    time_vector = np.arange(0, T_tot, dt)

    # FFT inv to get signal
    y_t = np.fft.irfft(y_f, axis=0, n=nfft_inv)  # (nt, nz, nr)
    y_t = np.real(y_t)  # Keep only real part

    # Build dataset to save
    ds_ir = xr.Dataset(
        coords=dict(
            t=time_vector,
            z=ds_tf_ir.z,
            r=ds_tf_ir.r,
        ),
        data_vars=dict(
            s=(["t", "z", "r"], y_t),
        ),
    )

    # Save dataset
    ds_ir.to_netcdf(ir_fpath)
else:
    ds_ir = xr.open_dataset(ir_fpath)

In [ ]:
z_plot = 995

r_offset = r_visu_ir[1] - r_visu_ir[0]
ts = ds_ir.t.diff("t").values[0]
tau_roll = (
    r_visu_ir - ds_ir.r.min().values
) / g.c0  # Roll time axis to start at zero
idx_tau_roll = tau_roll / ts
idx_tau_roll = idx_tau_roll.astype(int)

# Scale for visualization
max_amplitude = ds_ir.s.max().values
alpha_dilatation = 1 * r_offset
ds_ir["s"] = ds_ir.s / max_amplitude * alpha_dilatation

# Loop over range to visualize impulse response
plt.figure(figsize=(8, 12))
for ir, r in enumerate(r_visu_ir):
    # Get ir at range r
    sig = ds_ir.s.sel(r=r, z=z_plot, method="nearest") + (ir + 1) * r_offset

    # Roll over time axis for current range
    sig_roll = sig.roll(t=-idx_tau_roll[ir], roll_coords=False)
    sig_roll.plot(color="k")

plt.xlabel(f"Time  t - r/{g.c0} [s]")
plt.ylabel("Range [m]")

In [ ]:
z_plot = 995
r_plot = 30*1e3

ts = ds_ir.t.diff("t").values[0]
tau_roll = (r_plot - ds_ir.r.min().values) / g.c0  # Roll time axis to start at zero
idx_tau_roll = tau_roll / ts
idx_tau_roll = idx_tau_roll.astype(int)

sig = ds_ir.s.sel(r=r_plot, z=z_plot, method="nearest")
sig = sig / sig.max().values
sig_roll = sig.roll(t=-idx_tau_roll, roll_coords=False)
sig_roll.plot(color="k")
plt.xlabel(f"Time  t - r/{g.c0} [s]")
plt.ylabel("Amplitude")

## Analogie avec la méthode source image 

In [ ]:
# Represent image source for the 2 first groups
d = 20
z_src = 5
z_rcv = d-1
r_rcv = 100

m = 0 
z01 = 2 * d * m - z_src + z_rcv
z02 = 2 * d * (m + 1) - z_src - z_rcv
z03 = 2 * d * m + z_src + z_rcv
z04 = 2 * d * (m + 1) + z_src - z_rcv

m = 1
z11 = 2 * d * m - z_src + z_rcv
z12 = 2 * d * (m + 1) - z_src - z_rcv
z13 = 2 * d * m + z_src + z_rcv
z14 = 2 * d * (m + 1) + z_src - z_rcv

In [ ]:
# # Plot image source rays

# plt.figure()
# plt.ylim([-2*d, 2*d])
# plt.scatter([0, r_rcv], [z_src, z_rcv])
# plt.gca().invert_yaxis()

# # Surface interface
# plt.plot([0, r_rcv], [0, 0], 'k-', linewidth=1.5)
# # Bottom interface
# plt.plot([0, r_rcv], [d, d], 'k-', linewidth=1.5)

# # First group of image sources
# # Direct path
# plt.plot([0, r_rcv], [z_src, z_src + z01], label=r"$R_{01}$", color='C1')
# plt.plot([r_rcv, r_rcv, 0], [z_src + z01, z_src, z_src], color='C1', linestyle="--")

# # Reflexion on bottom
# plt.plot([0, r_rcv], [z_src, z_src + z02], label=r"$R_{02}$", color="C2")
# plt.plot(
#     [0, r_rcv], [d + (d - z_src), z_rcv], color="C2", linestyle="-"
# )
# plt.plot([r_rcv, r_rcv, 0], [z_src + z02, z_src, z_src], color='C2', linestyle="--")

# # Reflexion on surface
# plt.plot([0, r_rcv], [z_src, -z_rcv], label=r"$R_{03}$", color="C3")
# plt.plot(
#     [0, r_rcv], [-z_src, z_rcv], color="C3", linestyle="-"
# )
# # plt.plot([r_rcv, r_rcv, 0], [z_src + z03, z_src, z_src], color='C3', linestyle="--")

# # Surface + bottom
# plt.plot([0, r_rcv], [-z_src, 2*d-z_rcv], label=r"$R_{04}$", color="C4")
# # plt.plot(
# #     [0, r_rcv], [-z_src, d + (d - z_rcv)], color="C4", linestyle="-"
# # )
# plt.legend()
# # plt.xlim([r_rcv-1, r_rcv])

In [ ]:
# plt.figure()

# plt.scatter([0, r_rcv], [z_src, z_rcv])
# plt.gca().invert_yaxis()

# # Surface interface
# plt.plot([0, r_rcv], [0, 0], "k-", linewidth=1.5)
# # Bottom interface
# plt.plot([0, r_rcv], [d, d], "k-", linewidth=1.5)


# plt.plot([0]*8, [z01, z02, z03, z04, z11, z12, z13, z14], 'o')

In [ ]:
# # Direct arrival
# hd = depth - z_plot - src_depth
# rd = np.sqrt(r_plot**2 + hd**2)
# td = rd / g.c0
# print(f"Direct arrival time: {td:.2f} s")

print("Arrival :")
arrivals_true = print_arrivals(z_src=src_depth, z_rcv=z_plot, r=r_plot, depth=depth, n=1)
print("Reciprocity arrivals :")
arrivals_recip = print_arrivals(z_src=z_plot, z_rcv=src_depth, r=r_plot, depth=depth, n=1)

arrivals = arrivals_true

# arrivals = arrivals[1:, :]

In [ ]:
r_offset = 0 
# Apply delay
tau_roll = (r_plot - ds_ir.r.min().values - r_offset) / g.c0  # Roll time axis to start at zero
idx_tau_roll = tau_roll / ts
idx_tau_roll = idx_tau_roll.astype(int)


# Roll sig
sig = ds_ir.s.sel(r=r_plot, z=z_plot, method="nearest")
sig_roll = sig.roll(t=-idx_tau_roll, roll_coords=False)

# Delay arrivals
tau = (r_plot-r_offset) / g.c0
delayed_arrivals = arrivals - tau
print(f"Delayed arrivals: {delayed_arrivals}")

# # Direct arrival
# td_delayed = td - tau
# print(f"Delayed direct arrival time: {td_delayed:.2f} s")
# print(f"td - tr = {td - r_plot/g.c0:.2f} s")


In [ ]:
lf = LargeFigure()
plt.figure()
sig = ds_ir.s.sel(r=r_plot, z=z_plot, method="nearest")

# Normalize
max_amplitude = sig_roll.max().values
sig_roll = sig_roll / max_amplitude

sig_roll.plot(color="k")

for k in range(arrivals.shape[0]):
    plt.axvline(
        delayed_arrivals[k, 0],
        color="k",
        linestyle="--",
        linewidth=1,
    )
    plt.text(
        delayed_arrivals[k, 0] + 0.03,
        0.4,
        r"$t_{"
        + f"{k+1}"
        + r"} ="
        + f"{np.round(arrivals[k, 0], 1)}"
        + r"\, \textrm{s}$",
        rotation=90,
        color="k",
        fontsize=22,
    )

# # Add direct arrival 
# plt.axvline(
#     td_delayed,
#     color="r",
#     linestyle="--",
#     linewidth=1,
# )
# plt.text(
#     td_delayed + 0.03,
#     0.6,
#     r"$t_{d} ="
#     + f"{np.round(td, 1)}"
#     + r"\, \textrm{s}$",
#     rotation=90,
#     color="r",
#     fontsize=22,
# )

plt.xlabel(f"Time  t - r/{g.c0} [s]")
plt.ylabel("h(t)")
plt.ylim([-1.5, 1.5])
plt.xlim([0, 4])  # Cropp the end to avoid wrapped around signal artefacts 
# plt.legend()
plt.title("")
# Save figure as pdf
fpath = os.path.join(img_folder_path, f"{name}_ir_images_source_{r_plot:.0f}m.pdf")
plt.savefig(fpath)

## Durée de réverberation

$T_{60 \text{dB}}$ est défini comme le temps nécessaire pour que l'énergie acoustique décroisse de 60 dB 


Définition du SPL : 

$ SPL = 10 \log{ \frac{1}{p_0^2} \times \frac{1}{T} \int_0^{T} p(t)^2 dt}$

ou encore, 

$ SPL = 20 \log{ \frac{p_{rms}}{p_0} } $

avec, 

$p_{rms} \approx \sqrt{\frac{1}{K} \sum_{k=1}^{K} p[k]^2} $

In [ ]:
# Derive prms
p = sig_roll
t_win = 100
# print(f"Window size for RMS: {t_win} samples")
print(f"Window duration for RMS: {t_win * ts:.2f} s")
p2_roll = (p**2).rolling(t=t_win, center=True).mean()  
p_rms = np.sqrt(p2_roll)

# Plot p_rms
plt.figure()
p_rms.plot(color="k")
plt.xlabel(f"Time  t - r/{g.c0} [s]")
plt.ylabel("RMS pressure")
plt.title("")
plt.xlim([0, 4])  # Cropp the end to avoid wrapped around signal artefacts

In [ ]:
# Derive SPL
# p_rms /= p_rms.max().values
spl = 20 * np.log10(p_rms / g.p0)

# Plot spl
spl.plot(color="k")

# Add threshold line
threshold = -30  # Threshold in dB
th = np.max(spl).values + threshold
plt.axhline(
    y=th,
    color="r",
    linestyle="--",
    linewidth=1,
    label=f"{threshold} dB",
)


plt.xlim([0, 4])  # Cropp the end to avoid wrapped around signal artefacts
plt.xlabel(f"Time  t - r/{g.c0} [s]")
plt.ylabel(r"$L_p$ [dB re 1$\mu$Pa$^2$]")
plt.legend()
plt.title("")

# Save figure as pdf
fpath = os.path.join(img_folder_path, f"{name}_ir_Lp_{r_plot:.0f}m.pdf")
plt.savefig(fpath)

In [ ]:
# Derive tau_th 
tau_th = spl.t.where(spl < th).dropna("t").values[0]
print(f"Reverberation time ({threshold} dB) : {tau_th:.2f} s")

# Génération du jeux de données pour l'estimation des performances des méthodes de RTF 

## Paramètres de la source 

In [ ]:
output_fs = 4 * src_fs  # Output sampling frequency after propagation

# Number of snapshot desired to derive cov matrix
n_cov_snapshots = 10
# Cov snapshot lenght
cov_snapshot_duration = np.round(tau_th, 0)
cov_snapshot_size = int(
    cov_snapshot_duration * output_fs
)  # Number of samples in the time window to compute the covariance matrix
cov_snapshot_final_size = 2 ** int(np.log2(cov_snapshot_size) + 1)  # Next power of 2
effective_cov_snapshot_duration = cov_snapshot_final_size / output_fs

# Derive signal duration
src_signal_duration = effective_cov_snapshot_duration * n_cov_snapshots

print(f"Number of snapshots to derive cov matrix: {n_cov_snapshots}")
print(f"Cov snapshot duration: {effective_cov_snapshot_duration} s")
print(f"Source signal duration: {src_signal_duration} s")
print(f"Cov snapshot size: {cov_snapshot_final_size} samples")

In [ ]:
src_depth = depth - 1 # Reciprocity -> receiver depth 
src_min_freq = 0 # Hz
src_max_freq = 50 # Hz
src_fs = 100  # Hz

# Create dummy signal to make it easy to run kraken simulation
fc = 25
sg = SignalGenerator()
s, t = sg.pulse(T=src_signal_duration, fc=fc, fs=src_fs, t0=0)
s = sg.normalize_sig(s, normalize="max")

src = AcousticSource(
    signal=s,
    time=t,
    name="Pulse",
    waveguide_depth=depth,
    window=None,
    nfft=2 ** int(np.log2(s.size) + 1),
)

## Paramètres du domaine de calcul 

In [ ]:
max_range_km = 45
min_range_km = 15

## Définition du cas test avant calcul Kraken

In [ ]:
name = "perekis_short_ir_waveguide_perf"
title = "Pekeris waveguide with short impulse response - for RTF methods performance study"

# Common properties
zmin = 0
zmax = depth
rmax = max_range_km * 1e3
rmin = min_range_km * 1e3

min_phase_speed = 1000
max_phase_speed = 20000

bott_props = {
    "rho": rho_sediment * 1e-3,  # Density (g/cm^3)
    "c_p": c_sediment,  # P-wave celerity (m/s)
    "c_s": 0.0,  # S-wave celerity (m/s)
    "a_p": alpha_sediment,  # Compression wave attenuation (dB/wavelength)
    "a_s": 0.0,  # Shear wave attenuation (dB/wavelength)
}

# Set domain properties
domain_properties = DomainProperties(zmin=zmin, zmax=zmax, rmin=rmin, rmax=rmax, unit="m")

# Set source properties
src_properties = SourceProperties(
    src_type="point_source", src_depth=src_depth, freq=src.kraken_freq
)
# Set receiver properties : needs to cover the whole water domain
rcv_z_min = zmin
rcv_z_max = depth

# Number of receiver depths / ranges (flp file) : sufficient resolution for later use (can be easily downsampled afterwards)
dr = 50
dz = 5
nr_flp = int((rmax - rmin) / dr) + 1
nz_flp = int(rcv_z_max / dz) + 1

rcv_properties = ReceiverProperties(
    zmin=rcv_z_min, zmax=rcv_z_max, rmin=rmin, rmax=rmax, unit="m"
)

# Set kraken properties
nmedia = 2

top_hs = KrakenTopHalfspace(
    boundary_condition="vacuum",
    halfspace_properties=None,
    twersky_scatter_properties=None,
)

bott_hs = KrakenBottomHalfspace(
    boundary_condition="acousto_elastic",
    sigma=0.0,
    halfspace_properties=bott_props,
    fmin=src.kraken_freq.min(),
    alpha_wavelength=10,
)
# Set attenuation properties
att = KrakenAttenuation(units="dB_per_wavelength", use_volume_attenuation=False)
# Set SSP profile
z = [0, zmax]
c = [c_water, c_water]  # Constant celerity profile
ssp = SSPProfile(z=z, c=c)

# Create the medium = water column layer
medium = KrakenMedium(
    ssp_interpolation_method="C_linear",
    z_ssp=ssp.z,
    c_p=ssp.c,
    c_s=0.0,
    rho=1.0,
    a_p=0.0,
    a_s=0.0,
    nmesh=0,
    sigma=0.0,
)

bott_hs.derive_sedim_layer_max_depth(domain_properties.zmax_m)
max_rcv_depth = bott_hs.sedim_layer_max_depth
n_rcv_z = default_nb_rcv_z(
    fmax=src.kraken_freq.max(), max_depth=max_rcv_depth, n_per_l=10
)
field = KrakenField(
    phase_speed_limits=[min_phase_speed, max_phase_speed],
    src_depth=[src_properties.depth],
    n_rcv_z=n_rcv_z,
    rcv_z_min=0,
    rcv_z_max=max_rcv_depth,
    rcv_r_max=0.0,
)

kraken_properties = KrakenProperties(
    mode_coupling="coupled",
    mode_addition="coherent",
    n_mode=100,
    nr=nr_flp,
    nz=nz_flp,
    nmedia=nmedia,
    top_hs=top_hs,
    bott_hs=bott_hs,
    att=att,
    medium=medium,
    field=field,
)


k_tc = KrakenTestCase(
    name=name,
    title=title,
    root_dir=tc_root_dir,
    domain_properties=domain_properties,
    src_properties=src_properties,
    rcv_properties=rcv_properties,
    kraken_properties=kraken_properties,
)

In [ ]:
# Kraken compute otpion
run_kraken = False
# run_kraken = True

In [ ]:
if run_kraken:
    km = KrakenManager()

    # Because of the large number of frequencies we need to split the calculation into frequency chunks -> field does not work with broadband simulation using
    # nf > 1000 frequencies
    all_freqs = k_tc.src.freq
    chunk_size = 950  # Number of frequencies per chunk
    n_chunks = int(np.ceil(len(all_freqs) / chunk_size))
    pressure_field = []
    for ichunk in range(n_chunks):
        # Get the frequencies for the current chunk
        start_freq = ichunk * chunk_size
        end_freq = min((ichunk + 1) * chunk_size, len(all_freqs))
        freq_chunk = all_freqs[start_freq:end_freq]

        # Run Kraken for the current chunk
        print(f"Running Kraken for frequencies {start_freq} to {end_freq - 1} (chunk {ichunk + 1}/{n_chunks})")

        # update env freq 
        k_tc.env.freq = freq_chunk
        # Run Kraken and get the pressure field
        p, field_pos = km.runkraken(
            env=k_tc.env, flp=k_tc.flp, frequencies=freq_chunk
        )

        # Append the pressure field for the current chunk
        pressure_field.append(p)


    # Concatenate the pressure fields from all chunks
    pressure_field = np.concatenate(pressure_field, axis=0)


    # pressure_field, field_pos = km.runkraken(
    #     env=k_tc.env, flp=k_tc.flp, frequencies=k_tc.src.freq
    # )

In [ ]:
if run_kraken:
    # Store pressure field as netcdf using xarray
    pressure_field = np.squeeze(pressure_field)  # Remove singleton dimensions if any
    ds_tf = xr.Dataset(
        data_vars=dict(
            tf_real=(["f", "z", "r"], np.real(pressure_field)),
            tf_imag=(["f", "z", "r"], np.imag(pressure_field)),
        ),
        coords=dict(
            f=k_tc.src.freq,
            z=field_pos["r"]["z"],
            r=field_pos["r"]["r"],
        ),
        attrs=dict(
            title="Transfer functions for Pekeris waveguide",
            description="Transfer functions computed using Kraken for a Pekeris waveguide with short impulse response.",
            # date_created=np.datetime64("now"),
            note="Dataset for rtf performance analysis purpose: lower spatial resolution, reduced range coverage and long signal duration.",
            type="perf",
            fs=src_fs,
            signal_duration=src_signal_duration,
            waveguide_depth=depth,
            src_depth=src_depth,

        ),
    )
    # Save to netcdf
    tf_perf_fpath = os.path.join(data_folder_path, "tf_perf.nc")
    ds_tf.to_netcdf(tf_perf_fpath)
else:
    # Load transfer functions from netcdf
    ds_tf = xr.open_dataset(tf_perf_fpath)